# 검증된 원자료 기준 지역·연도 추세 시각화

- 이슈: #45
- 구조환경지표: 2016~2025년 검증본 21개 지표
- 계획예산: 2016~2024년 시행계획의 당해 명목 계획예산
- 분석 단위: 17개 시도. 전국 행은 QA·추세 비교 기준으로만 사용한다.
- 결측은 보간하거나 0으로 대체하지 않으며 그래프에서 선이 끊기도록 유지한다.
- 급등락 표시는 오류 확정이 아니라 원자료 재확인 후보(IQR 기준)다.


## 1. 공통 설정

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

np.random.seed(42)

repo_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / ".git").exists()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("현재 실행 위치의 상위 경로에서 Git 저장소를 찾지 못했습니다.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.features.trend_eda import (  # noqa: E402
    prepare_budget_trends,
    reshape_structural_indicators,
)
from src.visualization.plots import save_figure  # noqa: E402
from src.visualization.trends import (  # noqa: E402
    plot_budget_overview,
    plot_budget_region_small_multiples,
    plot_region_small_multiples,
    plot_structural_indicator_overview,
)

MAPPING_PATH = repo_root / "data" / "lookup" / "시도_지역코드_매핑.csv"
STRUCTURAL_PATH = (
    repo_root / "data" / "interim" / "구조환경지표_검증" / "구조환경지표_21개_검증본.csv"
)
BUDGET_PATH = (
    repo_root
    / "data"
    / "processed"
    / "analysis"
    / "2016-2024_시도별_계획예산_합계출산율_기초패널.csv"
)
OUTPUT_DIR = repo_root / "data" / "processed" / "eda" / "지역연도_추세"
FIGURE_DIR = repo_root / "notebooks" / "results" / "20260727_지역연도_추세"

region_order = pd.read_csv(MAPPING_PATH)["지역"].tolist()
assert len(region_order) == 17
print("저장소 루트:", repo_root)
print("시도:", region_order)

저장소 루트: /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha
시도: ['서울', '부산', '대구', '인천', '광주', '대전', '울산', '세종', '경기', '강원', '충북', '충남', '전북', '전남', '경북', '경남', '제주']


## 2. 구조환경지표 long 패널 및 QA

In [2]:
structural_wide = pd.read_csv(STRUCTURAL_PATH, encoding="utf-8-sig")
structural_long = reshape_structural_indicators(
    structural_wide,
    expected_regions=region_order,
)

structural_regions = structural_long.loc[~structural_long["지역"].eq("전국")]
structural_qa = pd.DataFrame(
    {
        "항목": [
            "long 패널 행 수",
            "세부지표 수",
            "시도 수",
            "지역×연도×지표 중복",
            "17개 시도 원자료 결측 셀",
            "급등락 후보 셀",
        ],
        "값": [
            len(structural_long),
            structural_long["세부지표"].nunique(),
            structural_regions["지역"].nunique(),
            int(structural_long.duplicated(["지역", "연도", "세부지표"]).sum()),
            int((~structural_regions["실측여부"]).sum()),
            int(structural_regions["급등락후보"].fillna(False).sum()),
        ],
    }
)
display(structural_qa)

,항목,값
0,long 패널 행 수,3770
1,세부지표 수,21
2,시도 수,17
3,지역×연도×지표 중복,0
4,17개 시도 원자료 결측 셀,992
5,급등락 후보 셀,125


In [3]:
structural_missing_summary = structural_regions.groupby(
    ["대영역", "세부영역", "세부지표"], as_index=False
).agg(
    전체셀=("실측여부", "size"),
    실측셀=("실측여부", "sum"),
)
structural_missing_summary["결측셀"] = (
    structural_missing_summary["전체셀"] - structural_missing_summary["실측셀"]
)
structural_missing_summary["결측률_pct"] = (
    structural_missing_summary["결측셀"].div(structural_missing_summary["전체셀"]).mul(100)
)

structural_outliers = structural_regions.loc[
    structural_regions["급등락후보"].fillna(False),
    ["지역", "연도", "대영역", "세부영역", "세부지표", "측정값", "전년대비변화", "검증상태"],
].sort_values(["세부지표", "연도", "지역"])

display(structural_missing_summary.sort_values("결측률_pct", ascending=False))
display(structural_outliers.head(30))

,대영역,세부영역,세부지표,전체셀,실측셀,결측셀,결측률_pct
11,3. 보건·안전,3-2. 산후조리 여건,산후조리원 보급도,170,51,119,70.000000
12,3. 보건·안전,3-2. 산후조리 여건,산후조리원 이용 요금,170,51,119,70.000000
20,4. 사회·문화,4-2. 사회적 가치관,출산에 대한 인식,170,68,102,60.000000
15,4. 사회·문화,4-1. 일·가정 양립 여건,가족친화인증기업 비율,170,68,102,60.000000
19,4. 사회·문화,4-2. 사회적 가치관,결혼에 대한 인식,170,85,85,50.000000
18,4. 사회·문화,4-2. 사회적 가치관,가사 분담에 대한 성평등 인식,170,85,85,50.000000
8,2. 가족·생활,2-2. 여가 인프라,여가생활 만족도,170,85,85,50.000000
1,1. 경제·고용·주거,1-3. 경제적 여건,소득만족도,170,85,85,50.000000
13,3. 보건·안전,3-3. 아동안전 수준,사회 안전에 대한 인식,170,85,85,50.000000
9,3. 보건·안전,3-1. 의료서비스 여건,분만실 병상수 보급도,170,136,34,20.000000


,지역,연도,대영역,세부영역,세부지표,측정값,전년대비변화,검증상태
358,충북,2024,4. 사회·문화,4-1. 일·가정 양립 여건,가족친화인증기업 비율,0.555325,0.095961,원자료 재현 완료했으나 2024년 대구·광주만 원자료 직접 카운트와 불일치(79건)...
594,대구,2020,4. 사회·문화,4-1. 일·가정 양립 여건,근로시간,157.200000,-24.400000,원자료 재현 검증 완료(오차 0)
646,울산,2022,4. 사회·문화,4-1. 일·가정 양립 여건,근로시간,169.200000,-9.900000,원자료 재현 검증 완료(오차 0)
801,세종,2017,2. 가족·생활,2-2. 여가 인프라,도시공원 보급도,84.100000,-18.100000,원자료가 로컬에 없어 라벨(정의)만 확인함 - 일부 지역·연도에서 30~100% 급...
861,제주,2017,2. 가족·생활,2-2. 여가 인프라,도시공원 보급도,5.400000,2.300000,원자료가 로컬에 없어 라벨(정의)만 확인함 - 일부 지역·연도에서 30~100% 급...
802,세종,2018,2. 가족·생활,2-2. 여가 인프라,도시공원 보급도,76.200000,-7.900000,원자료가 로컬에 없어 라벨(정의)만 확인함 - 일부 지역·연도에서 30~100% 급...
783,부산,2019,2. 가족·생활,2-2. 여가 인프라,도시공원 보급도,12.200000,5.300000,원자료가 로컬에 없어 라벨(정의)만 확인함 - 일부 지역·연도에서 30~100% 급...
803,세종,2019,2. 가족·생활,2-2. 여가 인프라,도시공원 보급도,69.000000,-7.200000,원자료가 로컬에 없어 라벨(정의)만 확인함 - 일부 지역·연도에서 30~100% 급...
854,전북,2020,2. 가족·생활,2-2. 여가 인프라,도시공원 보급도,24.400000,10.300000,원자료가 로컬에 없어 라벨(정의)만 확인함 - 일부 지역·연도에서 30~100% 급...
805,세종,2021,2. 가족·생활,2-2. 여가 인프라,도시공원 보급도,65.300000,-2.900000,원자료가 로컬에 없어 라벨(정의)만 확인함 - 일부 지역·연도에서 30~100% 급...


## 3. 구조환경지표 그래프 생성

전국 공표·산출값이 있으면 이를 기준선으로 사용하고, 근로시간처럼 전국값이 없으면 17개 시도 중앙값과 IQR을 표시한다.

In [4]:
figure_records = []
structural_figure_dir = FIGURE_DIR / "구조환경지표"
indicator_order = structural_wide["세부지표"].drop_duplicates().tolist()

for index, indicator in enumerate(indicator_order, start=1):
    stem = f"{index:02d}_{indicator}"

    overview = plot_structural_indicator_overview(
        structural_long,
        indicator=indicator,
    )
    overview_path = structural_figure_dir / f"{stem}_요약"
    save_figure(overview, overview_path)
    plt.close(overview)
    figure_records.append(
        {"구분": "구조환경지표 요약", "세부지표": indicator, "경로": str(overview_path)}
    )

    regional = plot_region_small_multiples(
        structural_long,
        indicator=indicator,
        region_order=region_order,
    )
    regional_path = structural_figure_dir / f"{stem}_17개시도"
    save_figure(regional, regional_path)
    plt.close(regional)
    figure_records.append(
        {"구분": "구조환경지표 지역", "세부지표": indicator, "경로": str(regional_path)}
    )

print(f"구조환경지표 그래프 세트: {len(figure_records)}개")

구조환경지표 그래프 세트: 42개


## 4. 계획예산 추세 및 QA

#53에서 생성한 검증 기초패널을 재사용한다. 금액은 실질화·인구 보정을 하지 않은 명목 계획예산이며 실제 집행액이 아니다.

In [5]:
if not BUDGET_PATH.exists():
    raise FileNotFoundError(
        f"#53 기초패널이 없습니다. 먼저 20260726 기초패널 생성 노트북을 실행하세요: {BUDGET_PATH}"
    )

budget_base = pd.read_csv(BUDGET_PATH, encoding="utf-8-sig")
budget_trends = prepare_budget_trends(
    budget_base,
    expected_regions=region_order,
)
budget_outliers = budget_trends.loc[
    budget_trends["급등락후보"],
    [
        "지역",
        "연도",
        "당해계획예산_백만원",
        "전년대비증감률_pct",
        "원자료_누락주의",
    ],
].sort_values(["연도", "지역"])

budget_annual_summary = budget_trends.groupby("연도", as_index=False).agg(
    **{"17개시도_합계_백만원": ("당해계획예산_백만원", "sum")},
    시도_중앙값_백만원=("당해계획예산_백만원", "median"),
    급등락후보수=("급등락후보", "sum"),
    원자료누락주의수=("원자료_누락주의", lambda values: int(values.notna().sum())),
)
display(budget_annual_summary)
display(budget_outliers)

,연도,17개시도_합계_백만원,시도_중앙값_백만원,급등락후보수,원자료누락주의수
0,2016,2.555166e+07,1198606.0,0,0
1,2017,3.100266e+07,1301513.8,2,0
2,2018,3.694802e+07,1629235.5,3,2
3,2019,3.707435e+07,1765898.0,3,0
4,2020,4.159559e+07,1730007.0,2,0
5,2021,4.672954e+07,1968341.0,2,0
6,2022,5.004321e+07,2258897.4,0,0
7,2023,5.784770e+07,2696432.0,2,0
8,2024,5.602570e+07,2621833.0,4,0


,지역,연도,당해계획예산_백만원,전년대비증감률_pct,원자료_누락주의
64,부산,2017,3477747.600,97.853748,NaN
91,울산,2017,2222729.400,285.124951,NaN
38,광주,2018,3239297.360,160.284710,NaN
92,울산,2018,948045.000,-57.347710,NaN
128,제주,2018,2211904.000,308.027346,NaN
39,광주,2019,1088029.000,-66.411574,NaN
93,울산,2019,356752.000,-62.369719,NaN
129,제주,2019,689085.000,-68.846523,NaN
40,광주,2020,194720.000,-82.103418,NaN
94,울산,2020,1240133.600,247.617841,NaN


In [6]:
budget_figure_dir = FIGURE_DIR / "계획예산"

budget_overview = plot_budget_overview(budget_trends)
budget_overview_path = budget_figure_dir / "계획예산_전국합계_지역분포_증감률_기본계획기간"
save_figure(budget_overview, budget_overview_path)
plt.close(budget_overview)
figure_records.append(
    {"구분": "계획예산 요약", "세부지표": pd.NA, "경로": str(budget_overview_path)}
)

budget_regional = plot_budget_region_small_multiples(
    budget_trends,
    region_order=region_order,
)
budget_regional_path = budget_figure_dir / "계획예산_17개시도_추세"
save_figure(budget_regional, budget_regional_path)
plt.close(budget_regional)
figure_records.append(
    {"구분": "계획예산 지역", "세부지표": pd.NA, "경로": str(budget_regional_path)}
)

print(f"전체 그래프 세트: {len(figure_records)}개")

전체 그래프 세트: 44개


## 5. 분석용 표와 그래프 목록 저장

In [7]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
figure_manifest = pd.DataFrame(figure_records)
figure_manifest["PNG"] = figure_manifest["경로"].astype(str) + ".png"
figure_manifest["PDF"] = figure_manifest["경로"].astype(str) + ".pdf"
figure_manifest = figure_manifest.drop(columns="경로")

outputs = {
    "구조환경지표_long.csv": structural_long,
    "구조환경지표_결측요약.csv": structural_missing_summary,
    "구조환경지표_급등락후보.csv": structural_outliers,
    "계획예산_지역연도_추세.csv": budget_trends,
    "계획예산_연도요약.csv": budget_annual_summary,
    "계획예산_급등락후보.csv": budget_outliers,
    "그래프_목록.csv": figure_manifest,
}
for file_name, frame in outputs.items():
    path = OUTPUT_DIR / file_name
    frame.to_csv(path, index=False, encoding="utf-8-sig")
    print(path.relative_to(repo_root), frame.shape)

display(figure_manifest)

data/processed/eda/지역연도_추세/구조환경지표_long.csv (3770, 13)
data/processed/eda/지역연도_추세/구조환경지표_결측요약.csv (21, 7)
data/processed/eda/지역연도_추세/구조환경지표_급등락후보.csv (125, 8)
data/processed/eda/지역연도_추세/계획예산_지역연도_추세.csv (153, 18)
data/processed/eda/지역연도_추세/계획예산_연도요약.csv (9, 5)
data/processed/eda/지역연도_추세/계획예산_급등락후보.csv (18, 5)
data/processed/eda/지역연도_추세/그래프_목록.csv (44, 4)


,구분,세부지표,PNG,PDF
0,구조환경지표 요약,청년고용률,/Users/leejungyeon/Workspace/projects/한국재...,/Users/leejungyeon/Workspace/projects/한국재...
1,구조환경지표 지역,청년고용률,/Users/leejungyeon/Workspace/projects/한국재...,/Users/leejungyeon/Workspace/projects/한국재...
2,구조환경지표 요약,소득만족도,/Users/leejungyeon/Workspace/projects/한국재...,/Users/leejungyeon/Workspace/projects/한국재...
3,구조환경지표 지역,소득만족도,/Users/leejungyeon/Workspace/projects/한국재...,/Users/leejungyeon/Workspace/projects/한국재...
4,구조환경지표 요약,소득수준,/Users/leejungyeon/Workspace/projects/한국재...,/Users/leejungyeon/Workspace/projects/한국재...
5,구조환경지표 지역,소득수준,/Users/leejungyeon/Workspace/projects/한국재...,/Users/leejungyeon/Workspace/projects/한국재...
6,구조환경지표 요약,보육시설 보급률,/Users/leejungyeon/Workspace/projects/한국재...,/Users/leejungyeon/Workspace/projects/한국재...
7,구조환경지표 지역,보육시설 보급률,/Users/leejungyeon/Workspace/projects/한국재...,/Users/leejungyeon/Workspace/projects/한국재...
8,구조환경지표 요약,방과후 돌봄시설 보급도,/Users/leejungyeon/Workspace/projects/한국재...,/Users/leejungyeon/Workspace/projects/한국재...
9,구조환경지표 지역,방과후 돌봄시설 보급도,/Users/leejungyeon/Workspace/projects/한국재...,/Users/leejungyeon/Workspace/projects/한국재...


## 해석상 주의사항

- 전국값과 17개 시도 단순평균은 혼용하지 않는다. 근로시간은 전국 공표값이 없어 시도 중앙값을 참고선으로 썼다.
- 명목 계획예산 합계는 전국 정부 총예산이 아니라 17개 시도 시행계획 세부사업의 관측 가능한 당해예산 합계다.
- 강원·전남 등 원자료 상세 누락 주의 지역은 실제 규모보다 과소대표될 수 있다.
- IQR 급등락 후보는 단위별 분포가 다른 지표의 재확인 목록이며 오류 확정값이 아니다.
- 제3차·제4차 기본계획 배경 구분은 기술적 시기 비교용이며 정책의 인과효과를 의미하지 않는다.
- 이 단계에서는 보간, 실질화, 대상인구 보정, 가중치 적용, 회귀분석을 수행하지 않는다.
